In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import urllib.request
from collections import deque

model_path = "pose_landmarker.task"
urllib.request.urlretrieve(
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task",
    model_path
)

base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    min_pose_detection_confidence=0.7,
    min_pose_presence_confidence=0.7,
    min_tracking_confidence=0.7,
    num_poses=4  #handle up to 4 people
)
landmarker = vision.PoseLandmarker.create_from_options(options)

#one deque per person slot for smoothing
SMOOTH_WINDOW = 10
torso_history = [deque(maxlen=SMOOTH_WINDOW) for _ in range(4)]

#skeleton connections
CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 7),          # face left
    (0, 4), (4, 5), (5, 6), (6, 8),          # face right
    (9, 10),                                 # mouth
    (11, 12),                                # shoulders
    (11, 13), (13, 15),                      # left arm
    (12, 14), (14, 16),                      # right arm
    (15, 17), (15, 19), (17, 19),            # left hand
    (16, 18), (16, 20), (18, 20),            # right hand
    (11, 23), (12, 24), (23, 24),            # torso
    (23, 25), (25, 27), (27, 29), (27, 31),  # left leg
    (24, 26), (26, 28), (28, 30), (28, 32),  # right leg
]

# distinct colors per person
PERSON_COLORS = [
    (0, 255, 0),
    (255, 100, 0),
    (0, 100, 255),
    (255, 0, 255),
]

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    results = landmarker.detect(mp_image)

    if results.pose_landmarks:
        for person_idx, landmarks in enumerate(results.pose_landmarks):
            color = PERSON_COLORS[person_idx % len(PERSON_COLORS)]

            pts = [(int(lm.x * w), int(lm.y * h)) for lm in landmarks]      # convert all landmarks to pixel coords

            #skeleton connections
            for a, b in CONNECTIONS:
                if landmarks[a].visibility > 0.5 and landmarks[b].visibility > 0.5:
                    cv2.line(frame, pts[a], pts[b], color, 2)

            #landmark dots
            for i, (px, py) in enumerate(pts):
                if landmarks[i].visibility > 0.5:
                    cv2.circle(frame, (px, py), 4, color, -1)
            for idx in [11, 12, 23, 24]:
                if landmarks[idx].visibility > 0.5:
                    cv2.circle(frame, pts[idx], 10, color, -1)

            #torso center x
            ls, rs, lh, rh = landmarks[11], landmarks[12], landmarks[23], landmarks[24]
            shoulder_cx = (ls.x + rs.x) / 2
            hip_cx = (lh.x + rh.x) / 2
            raw_torso_cx = ((shoulder_cx + hip_cx) / 2) * w

            #smooth using rolling average
            torso_history[person_idx].append(raw_torso_cx)
            smooth_torso_cx = int(np.mean(torso_history[person_idx]))

            #torso line
            cv2.line(frame, (smooth_torso_cx, 0), (smooth_torso_cx, h), color, 2)
            cv2.putText(frame, f"P{person_idx+1} X: {smooth_torso_cx}", (10, 40 + person_idx * 35),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

    else:
        cv2.putText(frame, "No pose detected", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    cv2.imshow("Pose Detection", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()